# Seahorse Quick Demo

Small interactive notebook for sanity-checking an installed `seahorse` wheel.

This notebook walks through:

- building a simple 1D transfer task
- solving the first few eigenstates
- propagating a state under a small control field
- evaluating a simple state-transfer fidelity
- optional plotting if `matplotlib` is available


## Prerequisite

Install a matching wheel first, for example:

```bash
python -m pip install bin/seahorse-0.1.0-...whl
```


In [ ]:
import numpy as np

import seahorse
from seahorse import (
    GridSpec,
    PotentialMode,
    PotentialSpec,
    TimeSpec,
    TransferTask,
    cosine_lattice,
    evaluate_control,
    propagate,
    solve_spectrum,
)

print("seahorse version:", seahorse.__version__)


In [ ]:
k = np.sqrt(2)
xlim = np.pi / k / 2 * 4

grid = GridSpec(dim=1 << 10, xmin=-xlim, xmax=xlim)
time = TimeSpec(dt=0.01, num_steps=160)

x = np.linspace(grid.xmin, grid.xmax, grid.dim)

base = cosine_lattice(x, depth=1000, wave_number=k, xlimit=(-np.pi/k/2, np.pi/k/2))

task = TransferTask(
    grid=grid,
    time=time,
    potential=PotentialSpec(base, mode=PotentialMode.SHIFTED),
    use_absorbing_boundary=False,
)

t = task.time.array()
control = 0.15 * np.sin(2.0 * np.pi * t[:-1] / t[-1])

print("grid dim:", task.grid.dim)
print("time steps:", task.time.num_steps)
print("control shape:", control.shape)


In [ ]:
spectrum = solve_spectrum(task, num_states=4)
print("eigenvalues:")
print(np.array2string(spectrum.eigenvalues, precision=6))

psi0 = np.asarray(spectrum.eigenvectors[:, 0], dtype=np.complex128)
psit = np.asarray(spectrum.eigenvectors[:, 1], dtype=np.complex128)

print("psi0 shape:", psi0.shape)
print("psit shape:", psit.shape)


In [ ]:
result = propagate(task, psi0, control, store_path=True)
evaluation = evaluate_control(task, psi0, psit, control)

final_overlap = np.vdot(psi0, result.final_state)

print("final_state_norm:", f"{np.linalg.norm(result.final_state):.6f}")
print("ground_state_overlap:", f"{abs(final_overlap):.6f}")
print("trajectory_shape:", result.trajectory.shape)
print("transfer_fidelity:", f"{evaluation.fid:.6f}")
print("transfer_cost:", f"{evaluation.cost:.6f}")


In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(x, base)
    axes[0].set_title("Base Potential")
    axes[0].set_xlabel("x")

    axes[1].plot(t[:-1], control)
    axes[1].set_title("Control")
    axes[1].set_xlabel("t")

    axes[2].plot(x, np.abs(result.trajectory[:, 0]) ** 2, label="initial")
    axes[2].plot(x, np.abs(result.trajectory[:, -1]) ** 2, label="final")
    axes[2].set_title("State Density")
    axes[2].set_xlabel("x")
    axes[2].legend()

    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib is not installed; skipping plots.")
